In [4]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import savgol_filter

def strong_valley_fix_with_tail(distances,
                                 valley_threshold=20.0,
                                 min_valley_len=2,
                                 sg_window=11,
                                 sg_poly=3,
                                 clip_min=10.0,
                                 clip_max=80.0,
                                 tail_len=3):
    data = np.array(distances, dtype=float)
    result = data.copy()
    n = len(data)

    # --- 深い谷の補完（線形補間） ---
    i = 1
    while i < n - 2:
        if result[i] < result[i - 1] - valley_threshold:
            start = i
            while i < n - 1 and result[i] < result[start - 1] - valley_threshold / 3:
                i += 1
            end = i
            if end - start >= min_valley_len and end < n:
                x = [start - 1, end]
                y = [result[start - 1], result[end]]
                interp_x = np.arange(start, end)
                result[start:end] = np.interp(interp_x, x, y)
        else:
            i += 1

    # --- Savitzky-Golay平滑化 ---
    if n >= sg_window:
        smoothed = savgol_filter(result, sg_window, sg_poly)
    else:
        smoothed = result.copy()

    # --- 末尾の傾きを使って末端補完（自然に減速） ---
    if n >= tail_len + 2:
        slope = smoothed[-tail_len - 1] - smoothed[-tail_len - 2]
        for i in range(1, tail_len + 1):
            smoothed[-i] = smoothed[-tail_len - 1] + slope * i

    # --- 値のクリップ（安全性） ---
    return np.clip(smoothed, clip_min, clip_max)

# ===== メイン処理 =====

with open("./testdistance_estimates.json", encoding="utf-8") as f:
    data = json.load(f)

scene_id = "043"
output_dir = "./scene3"
os.makedirs(output_dir, exist_ok=True)

if scene_id in data:
    frame_distances = data[scene_id]
    frames = sorted(frame_distances.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = [frame_distances[frame] for frame in frames]

    # 最終滑らか補正処理
    final_smoothed = strong_valley_fix_with_tail(distances)

    # グラフ描画
    plt.figure()
    plt.plot(range(1, len(final_smoothed)+1), final_smoothed, marker='o', label="Final Polished")
    plt.title(f"Scene {scene_id} - Distance Estimate (Fully Fixed)")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()

    save_path = os.path.join(output_dir, f"{scene_id}_fully_fixed.png")
    plt.savefig(save_path)
    plt.close()

    print(f"✅ Scene {scene_id} の完全補正グラフを保存しました: {save_path}")
else:
    print(f"⚠️ Scene {scene_id} はデータに存在しません。")


✅ Scene 043 の完全補正グラフを保存しました: ./scene3/043_fully_fixed.png


In [5]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

def ransac_polynomial_fill(distances, window_size=20, degree=2, threshold=2.0):
    """
    過去window_sizeフレームを使って、RANSAC + 多項式回帰で距離外れ値を補完する
    """
    data = np.array(distances, dtype=float)
    result = data.copy()
    n = len(data)

    for i in range(window_size, n):
        x_window = np.arange(i - window_size, i).reshape(-1, 1)
        y_window = data[i - window_size:i]

        if np.any(np.isnan(y_window)) or np.any(np.isinf(y_window)):
            continue

        model = make_pipeline(
            PolynomialFeatures(degree=degree),
            RANSACRegressor(residual_threshold=threshold)
        )
        model.fit(x_window, y_window)

        x_pred = np.array([[i]])
        y_pred = model.predict(x_pred)[0]

        if abs(data[i] - y_pred) > threshold:
            result[i] = y_pred

    return result

# ===== JSON 読み取りのみ =====

with open("./testdistance_estimates.json", encoding="utf-8") as f:
    data = json.load(f)

scene_id = "043"
output_dir = "./scene3"
os.makedirs(output_dir, exist_ok=True)

if scene_id in data:
    # フレームごとの距離リストを取得
    frame_data = data[scene_id]
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = [frame_data[key] for key in frame_keys]

    # 補完処理
    corrected = ransac_polynomial_fill(distances, window_size=20, degree=2, threshold=2.0)

    # グラフ描画（保存のみ）
    plt.figure()
    plt.plot(distances, label="Original", alpha=0.4, marker='o')
    plt.plot(corrected, label="RANSAC Corrected", alpha=0.9, marker='x')
    plt.title(f"Scene {scene_id} - RANSAC Polynomial Correction")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.legend()
    plt.grid(True)

    save_path = os.path.join(output_dir, f"{scene_id}_ransac_corrected_graph.png")
    plt.savefig(save_path)
    plt.close()

    print(f"✅ RANSAC補完後のグラフを保存しました（JSON未変更）: {save_path}")
else:
    print(f"⚠️ Scene {scene_id} は JSON に存在しません。")


/home/kimura2003/subaru/project/project2/project3/yolonas_env/lib/python3.9/site-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/kimura2003/subaru/project/project2/project3/yolonas_env/lib/python3.9/site-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/kimura2003/subaru/project/project2/project3/yolonas_env/lib/python3.9/site-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/kimura2003/subaru/project/project2/project3/yolonas_env/lib/python3.9/site-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, Undefine

✅ RANSAC補完後のグラフを保存しました（JSON未変更）: ./scene3/043_ransac_corrected_graph.png


In [8]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

def ransac_polynomial_fill(distances, window_size=30, degree=2, threshold=4.0,
                           clip_min=10.0, clip_max=80.0):
    """
    RANSAC + 多項式回帰で距離外れ値を補完する（補完点を増やしつつ異常補完を防ぐ）
    """
    data = np.array(distances, dtype=float)
    result = data.copy()
    mask = np.zeros_like(data, dtype=bool)  # 補完された点のマスク
    n = len(data)

    for i in range(window_size, n):
        x_window = np.arange(i - window_size, i).reshape(-1, 1)
        y_window = data[i - window_size:i]

        if np.any(np.isnan(y_window)) or np.any(np.isinf(y_window)):
            continue

        model = make_pipeline(
            PolynomialFeatures(degree=degree),
            RANSACRegressor(residual_threshold=threshold, max_trials=1000, min_samples=0.5)
        )

        try:
            model.fit(x_window, y_window)
            y_pred = model.predict(np.array([[i]]))[0]

            # 補完条件＋クリップ制限
            if abs(data[i] - y_pred) > threshold and clip_min <= y_pred <= clip_max:
                result[i] = y_pred
                mask[i] = True
        except Exception:
            continue

    return np.clip(result, clip_min, clip_max), mask

# ===== JSON 読み取りのみ =====

with open("./testdistance_estimates.json", encoding="utf-8") as f:
    data = json.load(f)

scene_id = "043"
output_dir = "./scene3"
os.makedirs(output_dir, exist_ok=True)

if scene_id in data:
    frame_data = data[scene_id]
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = [frame_data[key] for key in frame_keys]

    # 補完処理（マスクも返す）
    corrected, mask = ransac_polynomial_fill(distances)

    # グラフ描画（保存のみ）
    plt.figure(figsize=(10, 6))
    plt.plot(distances, label="Original", alpha=0.4, marker='o')
    plt.plot(corrected, label="RANSAC Corrected", alpha=0.9, marker='x')
    plt.scatter(np.where(mask)[0], corrected[mask], color="red", label="補完点", zorder=5)

    plt.title(f"Scene {scene_id} - RANSAC Polynomial Correction (Improved)")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()

    save_path = os.path.join(output_dir, f"{scene_id}_ransac_corrected_graph.png")
    plt.savefig(save_path)
    plt.close()

    print(f"✅ 改善版RANSAC補完後のグラフを保存しました: {save_path}")
else:
    print(f"⚠️ Scene {scene_id} は JSON に存在しません。")


KeyboardInterrupt: 

In [9]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.exceptions import ConvergenceWarning
import warnings

# === 警告抑制（R^2未定義など） ===
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def ransac_polynomial_fill(distances,
                           window_size=30,
                           degree=2,
                           threshold=4.0,
                           clip_min=10.0,
                           clip_max=80.0):
    """
    RANSAC + 多項式回帰による距離補完（外れ値補完点をマスクで返す）
    """
    data = np.array(distances, dtype=float)
    result = data.copy()
    mask = np.zeros_like(data, dtype=bool)
    n = len(data)

    for i in range(window_size, n):
        if i % 20 == 0:
            print(f"Processing frame {i}/{n}")

        x_window = np.arange(i - window_size, i).reshape(-1, 1)
        y_window = data[i - window_size:i]

        if np.any(np.isnan(y_window)) or np.any(np.isinf(y_window)):
            continue

        model = make_pipeline(
            PolynomialFeatures(degree=degree),
            RANSACRegressor(
                residual_threshold=threshold,
                max_trials=100,
                stop_n_inliers=15,
                min_samples=0.5
            )
        )

        try:
            model.fit(x_window, y_window)
            y_pred = model.predict(np.array([[i]]))[0]

            if abs(data[i] - y_pred) > threshold and clip_min <= y_pred <= clip_max:
                result[i] = y_pred
                mask[i] = True
        except Exception as e:
            print(f"Frame {i}: RANSAC failed → {e}")
            continue

    return np.clip(result, clip_min, clip_max), mask

# ===== JSON 読み取り（変更なし） =====

with open("./testdistance_estimates.json", encoding="utf-8") as f:
    data = json.load(f)

scene_id = "043"
output_dir = "./scene3"
os.makedirs(output_dir, exist_ok=True)

if scene_id in data:
    frame_data = data[scene_id]
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = [frame_data[key] for key in frame_keys]

    # 補完実行
    corrected, mask = ransac_polynomial_fill(distances)

    # グラフ描画
    plt.figure(figsize=(10, 6))
    plt.plot(distances, label="Original", alpha=0.4, marker='o')
    plt.plot(corrected, label="RANSAC Corrected", alpha=0.9, marker='x')
    plt.scatter(np.where(mask)[0], corrected[mask], color="red", label="補完点", zorder=5)

    plt.title(f"Scene {scene_id} - RANSAC Polynomial Correction (Safe)")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()

    save_path = os.path.join(output_dir, f"{scene_id}_ransac_corrected_graph.png")
    plt.savefig(save_path)
    plt.close()

    print(f"✅ RANSAC補完後のグラフを保存しました: {save_path}")
else:
    print(f"⚠️ Scene {scene_id} は JSON に存在しません。")


Processing frame 40/131
Processing frame 60/131
Processing frame 80/131
Processing frame 100/131
Processing frame 120/131
✅ RANSAC補完後のグラフを保存しました: ./scene3/043_ransac_corrected_graph.png


In [14]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.exceptions import ConvergenceWarning
from scipy.signal import savgol_filter
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# === RANSAC補完 ===
def ransac_polynomial_fill(distances,
                           window_size=30,
                           degree=2,
                           threshold=4.0,
                           clip_min=10.0,
                           clip_max=80.0):
    data = np.array(distances, dtype=float)
    result = data.copy()
    mask = np.zeros_like(data, dtype=bool)
    n = len(data)

    for i in range(window_size, n):
        x_window = np.arange(i - window_size, i).reshape(-1, 1)
        y_window = data[i - window_size:i]

        if np.any(np.isnan(y_window)) or np.any(np.isinf(y_window)):
            continue

        model = make_pipeline(
            PolynomialFeatures(degree=degree),
            RANSACRegressor(
                residual_threshold=threshold,
                max_trials=100,
                stop_n_inliers=15,
                min_samples=0.5
            )
        )

        try:
            model.fit(x_window, y_window)
            y_pred = model.predict(np.array([[i]]))[0]
            if abs(data[i] - y_pred) > threshold and clip_min <= y_pred <= clip_max:
                result[i] = y_pred
                mask[i] = True
        except:
            continue

    return np.clip(result, clip_min, clip_max), mask

# === メイン処理 ===
with open("./testdistance_estimates.json", encoding="utf-8") as f:
    data = json.load(f)

scene_id = "122"
output_dir = "./scene3"
os.makedirs(output_dir, exist_ok=True)

if scene_id in data:
    frame_data = data[scene_id]
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = [frame_data[key] for key in frame_keys]

    # RANSAC補完（点単位）
    corrected, mask = ransac_polynomial_fill(distances)

    # === Savitzky-Golay 平滑化（全体に適用） ===
    # ウィンドウサイズは奇数・データ数より小さく、polyorder < window_size
    window_size = 15 if len(corrected) > 15 else (len(corrected) // 2) * 2 + 1
    smoothed = savgol_filter(corrected, window_length=window_size, polyorder=3)

    # グラフ描画
    plt.figure(figsize=(10, 6))
    plt.plot(distances, label="Original", alpha=0.4, marker='o')
    plt.plot(smoothed, label="Final Smoothed (Savgol)", alpha=0.9, marker='x')
    plt.scatter(np.where(mask)[0], smoothed[mask], color="red", label="補完点", zorder=5)

    plt.title(f"Scene {scene_id} - Final Smooth (Savgol)")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()

    save_path = os.path.join(output_dir, f"{scene_id}_savgol_smooth.png")
    plt.savefig(save_path)
    plt.close()

    print(f"✅ Savitzky-Golay補完後のグラフを保存しました: {save_path}")
else:
    print(f"⚠️ Scene {scene_id} は JSON に存在しません。")


✅ Savitzky-Golay補完後のグラフを保存しました: ./scene3/122_savgol_smooth.png


In [15]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# === 局所補完処理（安定区間から2次フィット） ===
def local_quadratic_interpolation(distances, start, end, margin=5):
    data = np.array(distances, dtype=float)
    result = data.copy()

    fit_start = max(0, start - margin)
    fit_end = min(len(data), end + margin)
    
    x_fit = np.arange(fit_start, fit_end)
    y_fit = result[fit_start:fit_end]

    # 安定区間から2次曲線フィット
    coeffs = np.polyfit(x_fit, y_fit, deg=2)
    x_interp = np.arange(start, end)
    y_interp = np.polyval(coeffs, x_interp)

    # 補完結果を反映
    result[start:end] = y_interp
    return result, x_interp, y_interp

# === JSON読み込み ===
with open("./testdistance_estimates.json", encoding="utf-8") as f:
    data = json.load(f)

scene_id = "122"
output_dir = "./scene3"
os.makedirs(output_dir, exist_ok=True)

if scene_id in data:
    frame_data = data[scene_id]
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = [frame_data[key] for key in frame_keys]

    # === 指定範囲を補完 ===
    start, end = 40, 70
    smoothed, interp_x, interp_y = local_quadratic_interpolation(distances, start, end)

    # === プロット ===
    plt.figure(figsize=(10, 6))
    plt.plot(distances, label="Original", alpha=0.4, marker='o')
    plt.plot(smoothed, label="Quadratic Locally Smoothed", alpha=0.9, marker='x')
    plt.scatter(interp_x, interp_y, color="red", label="補完", zorder=5)

    plt.title(f"Scene {scene_id} - Local Quadratic Interpolation ({start}〜{end})")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()

    save_path = os.path.join(output_dir, f"{scene_id}_quadratic_local_fit.png")
    plt.savefig(save_path)
    plt.close()

    print(f"✅ 区間補完グラフを保存しました: {save_path}")
else:
    print(f"⚠️ Scene {scene_id} が JSON に存在しません。")


✅ 区間補完グラフを保存しました: ./scene3/122_quadratic_local_fit.png


In [16]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline

def local_convex_spline_interpolation(distances, start, end, margin=5, smooth_factor=0.5):
    data = np.array(distances, dtype=float)
    result = data.copy()

    fit_start = max(0, start - margin)
    fit_end = min(len(data), end + margin)

    x_fit = np.arange(fit_start, fit_end)
    y_fit = result[fit_start:fit_end]

    # 上に凸にしたい → スプライン補完でなめらかにしつつ、丸みを維持
    spline = UnivariateSpline(x_fit, y_fit, s=smooth_factor * len(x_fit))
    x_interp = np.arange(start, end)
    y_interp = spline(x_interp)

    # 上に凸になるよう制限（前後の平均より高くないと却下）
    y_interp = np.maximum(y_interp, (result[start - 1] + result[end]) / 2)

    result[start:end] = y_interp
    return result, x_interp, y_interp

# === JSON読み込み ===
with open("./testdistance_estimates.json", encoding="utf-8") as f:
    data = json.load(f)

scene_id = "122"
output_dir = "./scene3"
os.makedirs(output_dir, exist_ok=True)

if scene_id in data:
    frame_data = data[scene_id]
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = [frame_data[key] for key in frame_keys]

    # === 指定区間をスプライン補完で上に凸に ===
    start, end = 40, 70
    smoothed, interp_x, interp_y = local_convex_spline_interpolation(distances, start, end)

    # === プロット ===
    plt.figure(figsize=(10, 6))
    plt.plot(distances, label="Original", alpha=0.4, marker='o')
    plt.plot(smoothed, label="Convex Spline Fit", alpha=0.9, marker='x')
    plt.scatter(interp_x, interp_y, color="red", label="補完", zorder=5)

    plt.title(f"Scene {scene_id} - Convex Spline Interpolation ({start}〜{end})")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()

    save_path = os.path.join(output_dir, f"{scene_id}_convex_spline_fit.png")
    plt.savefig(save_path)
    plt.close()

    print(f"✅ 上に凸のスプライン補完グラフを保存しました: {save_path}")
else:
    print(f"⚠️ Scene {scene_id} が JSON に存在しません。")


✅ 上に凸のスプライン補完グラフを保存しました: ./scene3/122_convex_spline_fit.png


In [20]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

def global_curve_fit(distances, start, end, margin=5):
    """
    frame[start:end] を、start-margin〜end+margin の滑らかな曲線で補完。
    補完関数は全区間を通る上に凸の曲線（3次多項式）とする。
    """
    data = np.array(distances, dtype=float)
    result = data.copy()

    fit_start = max(0, start - margin)
    fit_end = min(len(data), end + margin)

    x_fit = np.arange(fit_start, fit_end)
    y_fit = result[fit_start:fit_end]

    # なだらかな上に凸の曲線を作るために3次フィット
    coeffs = np.polyfit(x_fit, y_fit, deg=3)
    poly = np.poly1d(coeffs)

    x_interp = np.arange(start, end)
    y_interp = poly(x_interp)

    result[start:end] = y_interp
    return result, x_fit, poly(x_fit), x_interp, y_interp

# === JSON読み込み ===
with open("./testdistance_estimates.json", encoding="utf-8") as f:
    data = json.load(f)

scene_id = "122"
output_dir = "./scene3"
os.makedirs(output_dir, exist_ok=True)

if scene_id in data:
    frame_data = data[scene_id]
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = [frame_data[key] for key in frame_keys]

    # === なだらかな谷 or 山で補完（画像に近づける） ===
    start, end = 40, 70
    smoothed, fit_x, fit_y, interp_x, interp_y = global_curve_fit(distances, start, end, margin=5)

    # === グラフ描画 ===
    plt.figure(figsize=(10, 6))
    plt.plot(distances, label="Original", alpha=0.4, marker='o')
    plt.plot(smoothed, label="Global 3rd-Degree Fit", alpha=0.9, marker='x')
    plt.plot(fit_x, fit_y, color="black", linewidth=2, label="補完曲線（全体）")
    plt.scatter(interp_x, interp_y, color="red", label="補完点", zorder=5)

    plt.title(f"Scene {scene_id} - Convex Global Fit ({start}〜{end})")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()

    save_path = os.path.join(output_dir, f"{scene_id}_global_fit_convex.png")
    plt.savefig(save_path)
    plt.close()

    print(f"✅ 画像のような滑らかな補完カーブを出力しました: {save_path}")
else:
    print(f"⚠️ Scene {scene_id} が JSON に存在しません。")


✅ 画像のような滑らかな補完カーブを出力しました: ./scene3/122_global_fit_convex.png


In [23]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from scipy.interpolate import make_interp_spline

# --- 異常値検出：最頻値割合が低い点 ---
def detect_outlier_indices(distances, window_size=3, threshold_ratio=0.2):
    outlier_indices = []
    for i in range(len(distances)):
        left = max(0, i - window_size)
        right = min(len(distances), i + window_size + 1)
        window = distances[left:right]
        counts = Counter(window)
        mode_val, count = counts.most_common(1)[0]
        if count / len(window) < threshold_ratio:
            outlier_indices.append(i)
    return outlier_indices

# --- 平滑化：指数移動平均 ---
def exponential_smoothing(data, alpha=0.3):
    smoothed = [data[0]]
    for v in data[1:]:
        smoothed.append(alpha * v + (1 - alpha) * smoothed[-1])
    return smoothed

# --- 急変補正：前フレームとの差が10以上なら前値を引き継ぐ ---
def apply_jump_suppression(data, threshold=10.0):
    corrected = data.copy()
    for i in range(1, len(corrected)):
        if abs(corrected[i] - corrected[i - 1]) > threshold:
            corrected[i] = corrected[i - 1]
    return corrected

# --- ベジェ風スプライン補間 ---
def bezier_like_spline(x, y, num=200):
    spline = make_interp_spline(x, y, k=3)
    x_new = np.linspace(x[0], x[-1], num)
    y_new = spline(x_new)
    return x_new, y_new

# === メイン ===
with open("./testdistance_estimates.json", encoding="utf-8") as f:
    data = json.load(f)

scene_id = "122"
output_dir = "./scene_bezier_jumpfilter"
os.makedirs(output_dir, exist_ok=True)

if scene_id in data:
    # --- 元データ読み込み ---
    frame_data = data[scene_id]
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = [frame_data[key] for key in frame_keys]
    distances_np = np.array(distances, dtype=float)

    # --- 異常検出 ---
    outlier_indices = detect_outlier_indices(distances_np)

    # --- 異常値補正（移動平均） ---
    smoothed_initial = exponential_smoothing(distances_np)

    corrected = distances_np.copy()
    for idx in outlier_indices:
        corrected[idx] = smoothed_initial[idx]

    # --- 急変補正（±10以上は前値コピー）---
    corrected = apply_jump_suppression(corrected, threshold=10.0)

    # --- スプライン補完 ---
    x = np.arange(len(corrected))
    x_spline, y_spline = bezier_like_spline(x, corrected, num=len(corrected))

    # --- グラフ描画 ---
    plt.figure(figsize=(10, 6))
    plt.plot(distances_np, label="Original", color="blue", alpha=0.4, marker='o')
    plt.plot(corrected, label="Jump-corrected", color="orange", marker='x')
    plt.plot(x_spline, y_spline, label="Bezier-like Smooth", color="green", linewidth=2)
    plt.scatter(outlier_indices, distances_np[outlier_indices], color="red", label="異常点", zorder=5)

    plt.title(f"Scene {scene_id} - Jump Suppression + Bezier Smooth")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()

    save_path = os.path.join(output_dir, f"{scene_id}_jump_filtered_bezier.png")
    plt.savefig(save_path)
    plt.close()

    print(f"✅ 補正済みグラフを保存しました: {save_path}")
else:
    print(f"⚠️ Scene {scene_id} が JSON に存在しません。")


✅ 補正済みグラフを保存しました: ./scene_bezier_jumpfilter/122_jump_filtered_bezier.png


In [25]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

# ---------- 設定 ----------
scene_id = "086"
json_path = "./testdistance_estimates.json"
output_dir = "./scene_bezier_jumpfilter"
os.makedirs(output_dir, exist_ok=True)

# ---------- ステップ1: JSON読み込み ----------
with open(json_path, encoding="utf-8") as f:
    data = json.load(f)

frame_data = data[scene_id]
frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
distances = np.array([frame_data[k] for k in frame_keys], dtype=float)

# ---------- ステップ2: 急激な変化を前の値に補正 ----------
def suppress_jumps(data, threshold=10.0):
    corrected = data.copy()
    for i in range(1, len(corrected)):
        if abs(corrected[i] - corrected[i - 1]) > threshold:
            corrected[i] = corrected[i - 1]
    return corrected

jump_corrected = suppress_jumps(distances)

# ---------- ステップ3: 安定区間のマスクを作成 ----------
def get_stable_indices(data, diff_threshold=3.0):
    diffs = np.abs(np.diff(data, prepend=data[0]))
    return np.where(diffs < diff_threshold)[0]

stable_indices = get_stable_indices(jump_corrected)
stable_x = stable_indices
stable_y = jump_corrected[stable_indices]

# ---------- ステップ4: 安定区間からスプライン補完 ----------
spline_model = CubicSpline(stable_x, stable_y, bc_type='natural')
x_all = np.arange(len(distances))
y_spline = spline_model(x_all)

# ---------- ステップ5: グラフ描画 ----------
plt.figure(figsize=(10, 6))
plt.plot(distances, label="Original", color="cornflowerblue", alpha=0.5, marker='o', markersize=4)
plt.plot(jump_corrected, label="Jump-corrected", color="orange", linestyle='--', marker='x', markersize=4)
plt.plot(x_all, y_spline, label="Smooth Spline", color="green", linewidth=2)
plt.title(f"Scene {scene_id} - Final Global Spline Fit")
plt.xlabel("Frame Index")
plt.ylabel("Distance (m)")
plt.grid(True)
plt.legend()
plt.tight_layout()

# ---------- ステップ6: 保存 ----------
save_path = os.path.join(output_dir, f"{scene_id}_global_spline_corrected.png")
plt.savefig(save_path)
plt.close()
print(f"✅ 補完グラフを保存しました: {save_path}")


✅ 補完グラフを保存しました: ./scene_bezier_jumpfilter/086_global_spline_corrected.png


In [26]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

# ---------- 設定 ----------
json_path = "./testdistance_estimates.json"
output_dir = "./scene_spline_all"
os.makedirs(output_dir, exist_ok=True)

# ---------- 関数：ジャンプ補正（10以上の差を前の値で補正） ----------
def suppress_jumps(data, threshold=10.0):
    corrected = data.copy()
    for i in range(1, len(corrected)):
        if abs(corrected[i] - corrected[i - 1]) > threshold:
            corrected[i] = corrected[i - 1]
    return corrected

# ---------- 関数：安定区間抽出 ----------
def find_stable_segments(data, diff_threshold=3.0, min_length=10):
    diffs = np.abs(np.diff(data, prepend=data[0]))
    stable_mask = diffs < diff_threshold

    segments = []
    start = None
    for i, val in enumerate(stable_mask):
        if val:
            if start is None:
                start = i
        else:
            if start is not None and i - start >= min_length:
                segments.append((start, i - 1))
            start = None
    if start is not None and len(data) - start >= min_length:
        segments.append((start, len(data) - 1))
    return segments

# ---------- 関数：安定区間からスプライン補完 ----------
def apply_spline_fit(x_all, data, stable_segments):
    stable_x = []
    stable_y = []
    for start, end in stable_segments:
        stable_x.extend(np.arange(start, end + 1))
        stable_y.extend(data[start:end + 1])
    if len(stable_x) < 4:
        return data  # スプラインに必要な点数未満
    spline = CubicSpline(stable_x, stable_y, bc_type='natural')
    return spline(x_all)

# ---------- メイン処理 ----------
with open(json_path, encoding="utf-8") as f:
    data = json.load(f)

for scene_id, frame_data in data.items():
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = np.array([frame_data[k] for k in frame_keys], dtype=float)

    # ステップ1：ジャンプ補正
    jump_corrected = suppress_jumps(distances)

    # ステップ2：安定区間検出
    segments = find_stable_segments(jump_corrected, diff_threshold=3.0, min_length=10)

    # ステップ3：スプライン補完
    x_all = np.arange(len(distances))
    smoothed = apply_spline_fit(x_all, jump_corrected, segments)

    # ステップ4：グラフ保存
    plt.figure(figsize=(10, 6))
    plt.plot(distances, label="Original", alpha=0.4, marker='o', markersize=3)
    plt.plot(jump_corrected, label="Jump-corrected", linestyle='--', marker='x', markersize=3)
    plt.plot(smoothed, label="Spline Smoothed", linewidth=2, color="green")
    plt.title(f"Scene {scene_id} - Spline Fit on Stable Segments")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    save_path = os.path.join(output_dir, f"{scene_id}_spline.png")
    plt.savefig(save_path)
    plt.close()

print("✅ 全シーンのスプライン補完グラフを保存しました。")


✅ 全シーンのスプライン補完グラフを保存しました。
